# Install Dependencies

In [ ]:
!pip install -q emoji
!pip install -q PyArabic
!pip install -q arabert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 11.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 11.0 MB/s eta 0:00:00


# Import Required Modules

In [ ]:
import pandas as pd
import re
import pyarabic.araby as araby
import string
import emoji

from transformers import AutoTokenizer, AutoModel
import torch

# Load Collected climate

In [ ]:
climate = pd.read_excel('All Climate Change Data - All Related.xlsx')

climate.drop_duplicates(subset='text', inplace = True)
climate.dropna(inplace = True, subset='text')
climate.reset_index(drop=True, inplace = True)

climate.shape

(23959, 6)

# Preprocessing

In [ ]:
climate['text'] = climate['text'].str.replace(r'[^\w\s]+', ' ')
climate['text'] = climate['text'].str.replace("\s+", " ", regex=True)

climate.head()

,text,Label 1,Label 2,Label 4,Label 3,Final Label
0,هذه ال٢.٥٪ لا تنطلق إلى الفضاء الكوني فتحتبس و...,Negative,Positive,Negative,Neutral,Negative
1,#عاجل | إدارة الكوارث والطوارئ التركية: إزالة ...,Negative,Neutral,Neutral,Positive,Neutral
2,RT @USUN: عُقد في مالطا هذا الأسبوع أول اجتماع...,Positive,Neutral,Neutral,Negative,Neutral
3,رغم ارتفاع درجات الحرارة وأشعة الشمس اللاهبة و...,Neutral,Positive,Positive,Negative,Positive
4,قبل أيام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Positive,Neutral,Neutral,Negative,Neutral


In [ ]:
arabic_punctuations = '''`÷×؛<>_()*&^%][ـ،/:"؟.,'{}~¦+|!”…“–ـ'''
english_punctuations = string.punctuation
punctuations_list = arabic_punctuations + english_punctuations

def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("گ", "ك", text)
    return text

In [ ]:
climate['text'] = climate['text'].apply(normalize_arabic)

climate.head()

,text,Label 1,Label 2,Label 4,Label 3,Final Label
0,هذه ال٢.٥٪ لا تنطلق الى الفضاء الكوني فتحتبس و...,Negative,Positive,Negative,Neutral,Negative
1,#عاجل | ادارة الكوارث والطوارئ التركية: ازالة ...,Negative,Neutral,Neutral,Positive,Neutral
2,RT @USUN: عُقد في مالطا هذا الاسبوع اول اجتماع...,Positive,Neutral,Neutral,Negative,Neutral
3,رغم ارتفاع درجات الحرارة واشعة الشمس اللاهبة و...,Neutral,Positive,Positive,Negative,Positive
4,قبل ايام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Positive,Neutral,Neutral,Negative,Neutral


In [ ]:
def data_cleaning(text):
    """Clean and preprocess text climate.
    Args:
        text (pd.Series): A pandas Series containing text climate to be cleaned.
    Returns:
        pd.Series: A pandas Series with the cleaned text climate.

    Cleaning Steps:
    - Removes emojis and special characters like '\x89Û_', '&amp', etc.
    - Replaces consecutive dots with an empty string.
    - Removes '#' symbol from text.
    - Removes user names starting with '@'.
    - Removes URLs starting with 'http' or 'https'.
    - Remove diacritics.
    - Remove English.
    - fix elongated words
    - Normalize Characters
    - Removes extra whitespaces between words.

    """
    clean = text
    # Replace consecutive dots with an empty string
    pattern = re.compile('\\.+?(?=\B|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Replace '\x89Û_' with a whitespace
    pattern = re.compile('\x89Û_')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Replace newline characters with a whitespace
    pattern = re.compile('\\n')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Remove '#' symbol from text
    clean = clean.apply(lambda r: r.replace('#', ''))
    # Remove '_' symbol from text
    pattern = re.compile('_')
    clean = clean.apply(lambda r: re.sub(pattern, ' ', r))
    # Replace user names with '@'
    pattern = re.compile('@[a-zA-Z0-9\_]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='@'))
    # Remove URLs
    pattern = re.compile('https?\S+(?=\s|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='www'))
    # Remove emojis
    clean = clean.apply(lambda r: emoji.replace_emoji(r, replace=""))
    # Remove diacritics
    clean = clean.apply(lambda r: araby.strip_diacritics(r))
    # Remove English
    # pattern = re.compile(r'[a-zA-Z]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Remove extra whitespaces
    clean = clean.apply(lambda r: ' '.join(r.split()))  # Remove extra whitespaces between words

    return clean

In [ ]:
climate['text'] = data_cleaning(climate['text'])

In [ ]:
def remove_ids(text):
  return text.split("—")[0].strip()

climate['text'] = climate['text'].apply(remove_ids)
climate.head()

,text,Label 1,Label 2,Label 4,Label 3,Final Label
0,هذه ال٢.٥٪ لا تنطلق الى الفضاء الكوني فتحتبس و...,Negative,Positive,Negative,Neutral,Negative
1,عاجل | ادارة الكوارث والطوارئ التركية: ازالة ا...,Negative,Neutral,Neutral,Positive,Neutral
2,RT @: عقد في مالطا هذا الاسبوع اول اجتماع لمجل...,Positive,Neutral,Neutral,Negative,Neutral
3,رغم ارتفاع درجات الحرارة واشعة الشمس اللاهبة و...,Neutral,Positive,Positive,Negative,Positive
4,قبل ايام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Positive,Neutral,Neutral,Negative,Neutral


In [ ]:
climate.drop_duplicates(subset='text', inplace = True)
climate.dropna(subset='text', inplace = True)
climate.shape

(23959, 6)

# Load Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02-twitter")
model = AutoModel.from_pretrained("aubmindlab/bert-base-arabertv02-twitter")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02-twitter and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Generate Embeddings for Collected climate

In [ ]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def tokenize_sentences(sentences):
    encoding = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
    # Move encoding tensors to GPU if available
    encoding = {k: v.to(device) for k, v in encoding.items()}
    return encoding

def get_average_word_embeddings(sentences):
    # Tokenize sentences
    encoding = tokenize_sentences(sentences)

    with torch.no_grad():
        # Get the model outputs
        outputs = model(**encoding)

        # Get the hidden states from the last layer (all token embeddings)
        last_hidden_states = outputs.last_hidden_state

        # For each sentence, calculate the mean of token embeddings (excluding padding tokens)
        attention_mask = encoding['attention_mask']

        # Create a mask to ignore padding tokens (padding tokens have attention mask 0)
        attention_mask = attention_mask.unsqueeze(-1).expand(last_hidden_states.size())

        # Sum embeddings and then normalize by the number of non-padding tokens
        sum_embeddings = torch.sum(last_hidden_states * attention_mask, dim=1)
        num_tokens = torch.sum(attention_mask, dim=1)

        # Avoid division by zero (in case of empty sentences or padding)
        avg_embeddings = sum_embeddings / num_tokens
        return avg_embeddings

In [ ]:
embeddings = get_average_word_embeddings(climate['text'].tolist())
embeddings = embeddings.cpu().numpy()

In [ ]:
embedding = pd.DataFrame(data = embeddings)
print(embedding.shape)
embedding.head()

(3959, 768)


,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,-0.357945,0.425827,0.007950,-1.101839,0.165160,0.217601,0.129883,0.944294,0.603453,0.919135,...,-1.019087,-0.214150,0.192232,-0.137054,0.290877,0.294218,-0.004254,0.126715,-0.251407,-0.019174
1,-0.643888,0.502055,0.292135,-0.555651,0.468832,0.439079,0.205376,0.749173,-0.201025,0.286383,...,-0.584945,0.814183,0.165179,0.383813,0.253561,0.404041,0.154517,0.210071,0.107585,-0.273468
2,-0.222225,0.552491,0.291763,-0.671435,0.288910,0.308418,-0.132264,0.715835,0.488945,0.535114,...,-0.107747,-0.358244,-0.003355,0.746578,0.221111,-0.720169,0.139077,0.532520,0.114829,-0.158362
3,-0.076294,0.697311,0.143696,-0.540779,0.461489,0.325071,0.299615,0.565882,-0.201597,0.620967,...,-0.076290,0.647311,0.110327,0.234448,0.098559,-0.057622,0.371208,0.659830,0.082804,0.109858
4,-0.157051,0.009081,0.269743,-1.047436,0.666985,0.064612,-1.405145,0.641826,0.193225,0.636512,...,0.081237,-0.051009,0.243680,0.217420,0.322247,-0.672687,0.217275,0.188078,-0.065459,0.388070


In [ ]:
embedding['Label'] = climate['Final Label']

In [ ]:
embedding.to_csv('Climate_AraBERTembeddings.csv', index = False)

# Load ASAD data

In [ ]:
asad = pd.read_csv('train_all.csv')

asad['Text'] = asad['Text'].str.replace(r'[^\w\s]+', '')
asad['Text'] = asad['Text'].str.replace("\s+", " ", regex=True)

asad['Text'] = asad['Text'].apply(normalize_arabic)

asad['Text'] = data_cleaning(asad['Text'])
asad['Text'] = asad['Text'].apply(remove_ids)

In [ ]:
asad.dropna(inplace = True)
asad = asad.drop_duplicates(subset='Text')
asad.shape

(53572, 3)

In [ ]:
embeddings = get_average_word_embeddings(asad['Text'].tolist())
embeddings = embeddings.cpu().numpy()

In [ ]:
embedding = pd.DataFrame(data = embeddings)
print(embedding.shape)
embedding.head()

(53691, 768)


,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,-0.326683,0.603533,-0.655818,-0.720379,0.388905,0.622459,1.593087,0.312437,-0.002240,0.546564,...,0.314739,0.239657,0.280964,0.536112,0.139862,1.422938,0.664465,0.428777,0.804685,0.062200
1,0.127981,0.617126,0.516041,-0.866202,0.440848,0.752861,-0.636583,0.667232,0.586483,0.275358,...,-0.167333,1.022732,0.311962,0.741841,0.699152,0.281070,0.277314,-0.391110,-0.180488,-0.027485
2,0.143708,0.469817,-0.111323,-0.094542,0.875489,0.456337,-0.264083,0.688284,-0.723363,0.407904,...,-0.966516,-0.161154,-0.204230,1.475959,0.258371,0.420809,0.320231,-0.012757,-0.288757,-0.249495
3,0.885687,0.915238,0.187001,-0.834882,0.774732,1.189938,0.750734,0.396684,0.281305,0.467433,...,-0.161310,0.700978,0.380903,0.336344,0.696964,1.089545,0.599843,0.370124,-0.260710,-0.443370
4,0.538969,0.994770,0.101932,0.152897,1.217631,0.781807,-1.422533,0.740518,-0.243590,-0.230382,...,-0.356478,-0.096432,0.445296,0.699667,-0.043325,0.986062,0.597686,0.421971,0.138051,-0.004959


In [ ]:
embedding['Label'] = asad['sentiment'].str.title()

In [ ]:
embedding.to_csv('ASAD_AraBERTembeddings.csv', index = False)

# Load ASTD dATA

In [ ]:
astd = pd.read_csv('Tweets.txt', delimiter='\t', header=None, names=['text', 'Label'])  # Change delimiter if needed
astd = astd[astd['Label']!='OBJ']
astd['Label'] = astd['Label'].map({
    'NEG': 'Negative',
    'NEUTRAL': 'Neutral',
    'POS': 'Positive'
})

astd['text'] = astd['text'].str.replace(r'[^\w\s]+', '')
astd['text'] = astd['text'].str.replace("\s+", " ", regex=True)

astd['text'] = astd['text'].apply(normalize_arabic)

astd['text'] = data_cleaning(astd['text'])
astd['text'] = astd['text'].apply(remove_ids)

astd.dropna(inplace = True)
astd = astd.drop_duplicates(subset = 'text')

In [ ]:
embeddings = get_average_word_embeddings(astd['text'].tolist())
embeddings = embeddings.cpu().numpy()

In [ ]:
embedding = pd.DataFrame(data = embeddings)
print(embedding.shape)
embedding.head()

(1611, 768)


,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,0.240154,0.326557,0.124639,-0.715556,0.875672,0.721680,2.297508,0.609392,-0.101960,0.024206,...,0.641436,0.349350,0.193320,0.776599,0.238353,0.425822,0.564655,0.452611,0.885924,-0.007880
1,-0.124997,0.562868,0.104761,-0.220216,0.704444,0.620704,0.251666,1.283273,0.600670,0.194405,...,-1.016036,0.492007,0.326659,1.034769,0.220343,0.066186,0.300825,0.691141,0.084252,0.058799
2,-0.287831,-0.312321,-0.271072,-0.535893,0.957960,0.343045,-0.128393,0.335230,0.265296,0.546230,...,0.179662,0.602812,0.119629,0.020693,0.360011,0.581414,-0.045916,0.156491,0.084337,0.264179
3,-0.027141,-0.058291,-0.003502,0.122458,-0.049465,0.530596,-1.346472,0.393827,0.896895,0.403341,...,-0.466384,-0.054126,-0.032793,0.216571,0.047988,-0.448855,0.318185,0.409114,-0.871395,0.049277
4,-0.022105,1.006474,0.347397,0.208448,0.580633,0.009919,-0.269782,0.401371,-0.360162,0.892615,...,-0.037565,-1.065326,-0.242106,1.519163,0.403830,1.471366,0.747151,0.091847,-0.777741,0.000130


In [ ]:
embedding['Label'] = astd['Label']

In [ ]:
embedding.to_csv('ASTD_AraBERTembeddings.csv', index = False)

# Load SemEval

In [ ]:
semeval = pd.read_csv('SemEval2017-task4-train.subtask-A.arabic.txt', delimiter='\t', header=None, names=['id', 'sentiment', 'text'])
semeval = semeval[['text', 'sentiment']]

semeval['sentiment'] = semeval['sentiment'].map({
    'negative': 'Negative',
    'neutral': 'Neutral',
    'positive': 'Positive'
})


semeval['text'] = semeval['text'].str.replace(r'[^\w\s]+', '')
semeval['text'] = semeval['text'].str.replace("\s+", " ", regex=True)

semeval['text'] = semeval['text'].apply(normalize_arabic)

semeval['text'] = data_cleaning(semeval['text'])
semeval['text'] = semeval['text'].apply(remove_ids)

semeval.dropna(inplace = True)
semeval = semeval.drop_duplicates(subset = 'text')


In [ ]:
semeval.shape

(3281, 2)

In [ ]:
embeddings = get_average_word_embeddings(semeval['text'].tolist())
embeddings = embeddings.cpu().numpy()

In [ ]:
embedding = pd.DataFrame(data = embeddings)
embedding.head()

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,0.183554,-0.036535,-0.219974,-0.704581,0.257537,-0.068995,-0.966787,0.305857,0.250942,-0.069977,...,-0.223710,0.901871,0.698177,-0.032965,0.280761,-0.023862,0.713233,0.801741,0.081938,-0.309413
1,0.452698,1.091138,-0.489178,-0.136248,0.983308,0.699742,-0.825917,0.389052,0.852679,-0.160084,...,-0.502223,0.558505,0.349227,0.504791,0.650556,0.342075,0.584451,0.202314,-0.381694,-0.149602
2,0.149042,1.583913,-0.182700,-0.334368,0.959656,1.217068,-0.997360,0.153490,0.705282,-0.094706,...,0.428007,0.203407,0.671057,0.752704,0.592532,1.034022,0.623807,0.232006,-0.307608,-0.544235
3,0.504900,0.104493,-0.159313,-0.451016,1.114642,1.168739,-0.985623,0.679405,0.569330,0.333545,...,0.014879,0.502601,0.666600,-0.279156,0.398753,-0.058021,0.810939,0.232892,-0.969782,-0.024017
4,0.135697,0.643511,-0.261706,0.311805,0.692465,-0.008097,-0.966501,0.512862,1.454186,-0.122026,...,-0.184593,0.855445,0.980271,0.311939,0.433502,-0.185125,0.182810,0.516094,-0.664540,0.400122


In [ ]:
embedding['Label'] = semeval['sentiment'].str.title()

In [ ]:
embedding.to_csv('SemEval_AraBERTembeddings.csv', index = False)

# Similarity between CLimate and ASTD

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df1 = pd.read_csv('/content/Climate_AraBERTembeddings.csv')
df1['Label'] = climate['Final Label'].values

df2 = pd.read_csv('/content/ASTD_AraBERTembeddings.csv')
df2['Label'] = astd['Label'].values

In [ ]:
neg1 = df1[df1['Label'] == 'Negative']
neg2 = df2[df2['Label'] == 'Negative']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(neg1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(neg2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between negative samples:", average_similarity)

Average cosine similarity between negative samples: 0.5417086702688261


In [ ]:
pos1 = df1[df1['Label'] == 'Positive']
pos2 = df2[df2['Label'] == 'Positive']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(pos1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(pos2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between positive samples:", average_similarity)

Average cosine similarity between positive samples: 0.5140895455861648


In [ ]:
neu1 = df1[df1['Label'] == 'Neutral']
neu2 = df2[df2['Label'] == 'Neutral']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(neu1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(neu2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between neutral samples:", average_similarity)

Average cosine similarity between neutral samples: 0.5282506286243802


In [ ]:
# Convert embeddings to numpy arrays
embeddings1 = np.vstack(df1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(df2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between all samples:", average_similarity)

Average cosine similarity between all samples: 0.5262926109455233


# Similarity Between Climate and ASAD

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df1 = pd.read_csv('/content/Climate_AraBERTembeddings.csv')
df1['Label'] = climate['Final Label'].values
df2 = pd.read_csv('/content/ASAD_AraBERTembeddings.csv')
df2['Label'] = asad['sentiment'].values

In [ ]:
df2['Label'] = df2['Label'].map(str.title)
df2.shape

(53572, 769)

In [ ]:
neg1 = df1[df1['Label'] == 'Negative']
neg2 = df2[df2['Label'] == 'Negative']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(neg1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(neg2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between negative samples:", average_similarity)

Average cosine similarity between negative samples: 0.5454675296265517


In [ ]:
pos1 = df1[df1['Label'] == 'Positive']
pos2 = df2[df2['Label'] == 'Positive']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(pos1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(pos2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between positive samples:", average_similarity)

Average cosine similarity between positive samples: 0.5127803055718413


In [ ]:
neu1 = df1[df1['Label'] == 'Neutral']
neu2 = df2[df2['Label'] == 'Neutral']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(neu1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(neu2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between neutral samples:", average_similarity)

Average cosine similarity between neutral samples: 0.5322068019011232


In [ ]:
# Convert embeddings to numpy arrays
embeddings1 = np.vstack(df1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(df2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between all samples:", average_similarity)

Average cosine similarity between all samples: 0.5324236308623197


# Similarity Between Climate and SemEval

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df1 = pd.read_csv('/content/Climate_AraBERTembeddings.csv')
df1['Label'] = climate['Final Label'].values

df2 = pd.read_csv('/content/SemEval_AraBERTembeddings.csv')
df2['Label'] =semeval['sentiment'].values

In [ ]:
neg1 = df1[df1['Label'] == 'Negative']
neg2 = df2[df2['Label'] == 'Negative']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(neg1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(neg2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between negative samples:", average_similarity)

Average cosine similarity between negative samples: 0.5334971453869429


In [ ]:
pos1 = df1[df1['Label'] == 'Positive']
pos2 = df2[df2['Label'] == 'Positive']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(pos1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(pos2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between positive samples:", average_similarity)

Average cosine similarity between positive samples: 0.5149133719122642


In [ ]:
neu1 = df1[df1['Label'] == 'Neutral']
neu2 = df2[df2['Label'] == 'Neutral']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(neu1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(neu2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between neutral samples:", average_similarity)

Average cosine similarity between neutral samples: 0.48901488287787315


In [ ]:
# Convert embeddings to numpy arrays
embeddings1 = np.vstack(df1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(df2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between all samples:", average_similarity)

Average cosine similarity between all samples: 0.5070551291795474
